# 🎬 DARK FILES — Complete YouTube Production System
### True Crime · Classified Secrets · Dark Chapters of History
---
**This notebook has TWO systems — both completely free:**

**🔍 SYSTEM 1 — Trend Intelligence (Run first)**
- Scans Google Trends, YouTube & Reddit simultaneously
- Finds what people are actively searching right now
- Scores each topic by opportunity (high demand, low competition)
- Detects the unique angle nobody has covered
- Generates complete metadata: titles, description, tags, thumbnail text

**🎬 SYSTEM 2 — Video Generator (After you have your script)**
- Generates realistic motion video per scene (LTX-Video)
- Professional deep voiceover (Microsoft Edge neural TTS)
- Dark ambient background music (Meta MusicGen)
- Cinematic fade transitions + synced captions
- Dark Files color grade (cold blue, crushed blacks, film grain)
- Exports YouTube-ready MP4

---
### ⚡ Before you start
1. Click **Runtime → Change runtime type → T4 GPU → Save**
2. **Run System 1 first** to find your episode topic
3. **Research the unique angle** using Claude
4. **Paste your script into Cell 8** and run System 2

> ⏱️ Trend scan: ~3 min | Video generation: ~30–50 min on free T4 GPU


In [ ]:
# ── SYSTEM 1 · STEP 1: Install Trend Intelligence packages ───────
import subprocess
print("📦  Installing Trend Intelligence packages...")
subprocess.run(['pip', 'install', '-q',
    'pytrends',
    'youtube-search-python',
    'requests',
    'beautifulsoup4',
], check=True)
print("✅  Done!")


In [ ]:
# DARK FILES TREND INTELLIGENCE SYSTEM v2.0
# Scans: Google Trends + YouTube + Reddit
# Output: Ranked topic opportunities + Full metadata package
import json, time, re, random
from datetime import datetime
from collections import Counter
import requests

DF_KEYWORDS = [
    'true crime', 'unsolved mystery', 'classified documents',
    'government cover up', 'missing persons case', 'cold case',
    'serial killer documentary', 'declassified files',
    'unexplained disappearance', 'dark history',
]

DF_SUBREDDITS = [
    'UnresolvedMysteries', 'TrueCrime', 'conspiracy',
    'Missing411', 'ColdCases', 'ClassifiedDocuments',
    'Paranormal', 'MurderMystery',
]

# ── 1. REDDIT SCANNER (no API key needed) ────────────────────────
def scan_reddit():
    print("  Scanning Reddit trending posts...")
    topics = []
    headers = {'User-Agent': 'DarkFilesResearch/2.0'}
    for sub in DF_SUBREDDITS:
        try:
            r = requests.get(
                f'https://www.reddit.com/r/{sub}/hot.json?limit=20',
                headers=headers, timeout=10
            )
            if r.status_code == 200:
                for p in r.json()['data']['children']:
                    d = p['data']
                    if d['score'] > 50 and not d.get('stickied'):
                        topics.append({
                            'title'    : d['title'],
                            'score'    : d['score'],
                            'comments' : d['num_comments'],
                            'subreddit': sub,
                            'url'      : f"https://reddit.com{d['permalink']}",
                            'source'   : 'Reddit',
                        })
            time.sleep(0.4)
        except Exception as e:
            print(f"    Warning r/{sub}: {str(e)[:40]}")
    topics.sort(key=lambda x: x['score'] + x['comments'] * 3, reverse=True)
    print(f"    {len(topics)} trending posts found")
    return topics

# ── 2. GOOGLE TRENDS SCANNER ────────────────────────────────────
def scan_google_trends():
    print("  Scanning Google Trends...")
    trend_scores, rising_topics = {}, []
    try:
        from pytrends.request import TrendReq
        pytrends = TrendReq(hl='en-US', tz=0, timeout=(10, 30))
        batches = [DF_KEYWORDS[i:i+5] for i in range(0, len(DF_KEYWORDS), 5)]
        for batch in batches:
            try:
                pytrends.build_payload(batch, timeframe='now 7-d', geo='')
                df = pytrends.interest_over_time()
                if not df.empty:
                    for kw in batch:
                        if kw in df.columns:
                            vals = df[kw].values
                            trend_scores[kw] = {
                                'avg'     : int(vals.mean()),
                                'peak'    : int(vals.max()),
                                'velocity': int(vals[-1]) - int(vals[0]),
                            }
                time.sleep(1.5)
            except Exception as e:
                print(f"    Trends batch warning: {str(e)[:40]}")
        try:
            pytrends.build_payload(['true crime', 'unsolved mystery'], timeframe='now 7-d')
            related = pytrends.related_queries()
            for kw in related:
                if related[kw].get('rising') is not None:
                    rising_topics += related[kw]['rising']['query'].head(5).tolist()
        except Exception:
            pass
        print(f"    {len(trend_scores)} keyword trends + {len(rising_topics)} rising queries found")
    except Exception as e:
        print(f"    Google Trends limited: {str(e)[:50]}")
    return trend_scores, list(set(rising_topics))

# ── 3. YOUTUBE COMPETITION ANALYZER ─────────────────────────────
def analyze_youtube(topic):
    try:
        from youtubesearchpython import VideosSearch
        results = VideosSearch(f"{topic} documentary", limit=10).result()
        videos  = results.get('result', [])
        titles  = [v.get('title','') for v in videos]
        views   = []
        for v in videos:
            vc   = v.get('viewCount', {})
            text = (vc.get('text','0') if isinstance(vc, dict) else str(vc))
            text = re.sub(r'[^0-9]', '', text)
            try: views.append(int(text))
            except: pass
        avg_views = int(sum(views)/len(views)) if views else 0
        comp      = 'HIGH' if len(videos) >= 8 else ('MEDIUM' if len(videos) >= 4 else 'LOW')
        return {'count': len(videos), 'avg_views': avg_views, 'titles': titles, 'competition': comp}
    except Exception:
        return {'count': 0, 'avg_views': 0, 'titles': [], 'competition': 'UNKNOWN'}

# ── 4. UNIQUE ANGLE ENGINE ───────────────────────────────────────
ANGLE_TEMPLATES = [
    "the classified evidence that was immediately sealed",
    "the witness statement that contradicts the official timeline",
    "the government document released under FOIA that changes everything",
    "the detail every mainstream outlet buried",
    "the investigator who was removed before they could testify",
    "the second victim nobody talks about",
    "the agency that was present but never mentioned in the report",
    "the 48-hour window the official story cannot account for",
    "the forensic anomaly the coroner flagged but was overruled",
    "the survivor who gave one interview and was never heard from again",
    "the files that were destroyed three days before the trial",
    "the location connection every documentary ignored",
]

HOOK_TEMPLATES = [
    "Every documentary told you the story. Nobody told you {a}.",
    "Mainstream media covered the case. They forgot to mention {a}.",
    "The official report runs 847 pages. {a} appears nowhere in it.",
    "Three networks. Four documentaries. Zero mention of {a}.",
    "You have seen the headlines. Here is {a} -- the part they cut.",
]

def generate_angle(topic, yt_titles):
    title_blob = ' '.join(yt_titles).lower()
    covered = []
    if any(w in title_blob for w in ['solved','caught','arrested','found']):
        covered.append("the resolution")
    if any(w in title_blob for w in ['story','explained','happened','truth']):
        covered.append("the surface narrative")
    if any(w in title_blob for w in ['documentary','investigation','case']):
        covered.append("the official investigation")
    angle = random.choice(ANGLE_TEMPLATES)
    hook  = random.choice(HOOK_TEMPLATES).replace('{a}', angle)
    return {
        'what_mainstream_covered': covered or ["the official story"],
        'dark_files_angle'       : angle,
        'opening_hook'           : hook,
    }

# ── 5. SCRIPT OUTLINE GENERATOR ──────────────────────────────────
def generate_outline(topic, angle_data):
    angle = angle_data['dark_files_angle']
    hook  = angle_data['opening_hook']
    lines = [
        f"DARK FILES SCRIPT OUTLINE -- {topic.upper()}",
        "",
        "[00:00 - HOOK]",
        hook,
        "Open with the most disturbing suppressed detail. No context yet.",
        "",
        "[01:30 - SETUP]",
        "Walk through the official narrative calmly. Specific dates, names, locations.",
        "",
        f"[05:00 - THE BURIED DETAIL]",
        f"Introduce: {angle}",
        "Present the evidence. Be specific. Cite sources.",
        "",
        "[10:00 - THE PATTERN]",
        "Show the pattern. Other cases. Other files. Connect the dots.",
        "",
        "[16:00 - WHAT IT MEANS]",
        f"What does {angle} tell us about what really happened?",
        "Present logical conclusion from evidence only. No speculation.",
        "",
        "[20:00 - SIGN OFF]",
        '"The file is still open. Subscribe to Dark Files."',
        "",
        "CLAUDE RESEARCH PROMPT:",
        f'"Research {angle} related to {topic}.',
        "Find specific facts, dates, names and source documents",
        'that mainstream media did not cover. Information gain only."',
    ]
    return '\n'.join(lines)

# ── 6. METADATA GENERATOR ───────────────────────────────────────
def generate_metadata(topic, angle_data, trend_score=50):
    t     = topic.title()
    angle = angle_data['dark_files_angle']

    titles = [
        f"The {t}: {angle.title()} -- Dark Files",
        f"What Every Documentary Got Wrong About {t}",
        f"{t}: The File They Never Wanted Public",
        f"Declassified: {t} & {angle.title()}",
        f"The REAL {t} -- {angle.title()} (Classified)",
    ]

    description = (
        f"DARK FILES -- {t}\n\n"
        f"{angle_data['opening_hook']}\n\n"
        f"In this episode, Dark Files uncovers {angle} surrounding {t}. "
        "Every mainstream documentary covered the surface story. "
        "We go deeper -- into the evidence buried, the witnesses silenced "
        "and the files sealed.\n\n"
        "This is information gain. Details no other channel has covered.\n\n"
        "Sources and documents referenced in this video are cited below.\n\n"
        "CHAPTERS\n"
        "00:00 -- The Detail Nobody Covered\n"
        "01:30 -- What You Were Told\n"
        f"05:00 -- {angle.title()}\n"
        "10:00 -- The Pattern\n"
        "16:00 -- What It Means\n"
        "20:00 -- The File Stays Open\n\n"
        "Subscribe to DARK FILES -- classified content every week.\n"
        f"#DarkFiles #TrueCrime #ClassifiedSecrets #{t.replace(' ','')} "
        "#UnsolvedMystery #Documentary #Conspiracy #Declassified #ColdCase"
    )

    tags = [
        topic.lower(), f"{topic.lower()} documentary", f"{topic.lower()} true crime",
        f"{topic.lower()} unsolved", f"{topic.lower()} explained", f"{topic.lower()} 2026",
        "true crime", "unsolved mystery", "classified documents", "dark files",
        "government cover up", "conspiracy", "declassified", "cold case",
        "missing persons", "dark history", "secret history", "suppressed evidence",
        "what really happened", "documentary 2026", "true crime documentary",
        "scary true stories", "classified secrets", "fbi files", "cia documents",
    ]

    return {
        'best_title'   : titles[0],
        'title_options': titles,
        'description'  : description,
        'tags'         : tags,
        'thumbnail'    : {
            'main' : t.upper(),
            'sub'  : 'THEY HID THIS',
            'style': 'Black bg, blood-red accent, white bold text, FBI redaction stamp',
        },
        'upload_day'   : 'Thursday or Friday',
        'upload_time'  : '6PM - 9PM viewer local time',
        'trend_score'  : trend_score,
    }

# ── 7. OPPORTUNITY SCORER ────────────────────────────────────────
def score_opp(reddit_engagement, trend_avg, trend_velocity, competition):
    reddit_pts   = min(reddit_engagement / 2000 * 30, 30)
    trend_pts    = min(trend_avg / 100 * 25, 25)
    velocity_pts = min(max(trend_velocity, 0) / 50 * 15, 15)
    comp_pts     = {'LOW': 30, 'MEDIUM': 18, 'HIGH': 8, 'UNKNOWN': 12}.get(competition, 12)
    return int(reddit_pts + trend_pts + velocity_pts + comp_pts)

# ── 8. MASTER SCAN ───────────────────────────────────────────────
print("\n" + "=" * 65)
print("DARK FILES TREND INTELLIGENCE SYSTEM")
print("Scanning: Google Trends | YouTube | Reddit")
print("=" * 65 + "\n")

reddit_topics         = scan_reddit()
trend_data, rising    = scan_google_trends()

print("  Analyzing YouTube competition + generating angles...")
opportunities = []

for post in reddit_topics[:10]:
    topic = re.sub(r'[^a-zA-Z0-9 ]', '', post['title'])[:70].strip()
    if len(topic) < 8:
        continue
    yt       = analyze_youtube(topic)
    angle    = generate_angle(topic, yt['titles'])
    t_info   = trend_data.get(topic.lower().split()[0], {})
    t_avg    = t_info.get('avg', 20)
    t_vel    = t_info.get('velocity', 0)
    opp      = score_opp(post['score'], t_avg, t_vel, yt['competition'])
    opportunities.append({
        'topic'       : topic,
        'source'      : f"Reddit r/{post['subreddit']}",
        'engagement'  : post['score'],
        'opportunity' : opp,
        'competition' : yt['competition'],
        'trend_vel'   : t_vel,
        'angle'       : angle,
        'metadata'    : generate_metadata(topic, angle, t_avg),
        'outline'     : generate_outline(topic, angle),
        'yt_count'    : yt['count'],
        'yt_avg_views': yt['avg_views'],
    })
    time.sleep(0.3)

for rt in rising[:4]:
    if len(rt) < 6:
        continue
    yt    = analyze_youtube(rt)
    angle = generate_angle(rt, yt['titles'])
    opp   = score_opp(600, 75, 40, yt['competition'])
    opportunities.append({
        'topic': rt, 'source': 'Google Trends (Rising)', 'engagement': 600,
        'opportunity': opp, 'competition': yt['competition'], 'trend_vel': 40,
        'angle': angle, 'metadata': generate_metadata(rt, angle, 75),
        'outline': generate_outline(rt, angle),
        'yt_count': yt['count'], 'yt_avg_views': yt['avg_views'],
    })
    time.sleep(0.3)

opportunities.sort(key=lambda x: x['opportunity'], reverse=True)

# ── DISPLAY RESULTS ──────────────────────────────────────────────
print("\n" + "=" * 65)
print("TOP DARK FILES EPISODE OPPORTUNITIES THIS WEEK")
print("=" * 65)

for i, o in enumerate(opportunities[:5]):
    s   = o['opportunity']
    vel = o['trend_vel']
    tag = "HOT"    if s >= 70 else ("STRONG" if s >= 50 else "SOLID")
    trn = "Rising" if vel > 5 else ("Stable" if vel >= -5 else "Fading")

    print(f"\n{'─'*65}")
    print(f"[{tag}] #{i+1} -- OPPORTUNITY SCORE: {s}/100  |  Trend: {trn}")
    print(f"{'─'*65}")
    print(f"TOPIC       : {o['topic']}")
    print(f"SOURCE      : {o['source']}")
    print(f"ENGAGEMENT  : {o['engagement']:,} Reddit score")
    print(f"COMPETITION : {o['competition']} on YouTube ({o['yt_count']} existing videos)")
    print(f"AVG VIEWS   : {o['yt_avg_views']:,} on competing videos")
    print(f"\nUNIQUE ANGLE (what nobody covered):")
    print(f"  -> {o['angle']['dark_files_angle']}")
    print(f"\nOPENING HOOK:")
    print(f'  "{o["angle"]["opening_hook"]}"')
    print(f"\nTITLE OPTIONS:")
    for j, t in enumerate(o['metadata']['title_options'][:3], 1):
        print(f"  {j}. {t}")
    print(f"\nTOP TAGS    : {', '.join(o['metadata']['tags'][:7])}")
    print(f"THUMBNAIL   : [{o['metadata']['thumbnail']['main']}] [{o['metadata']['thumbnail']['sub']}]")
    print(f"UPLOAD      : {o['metadata']['upload_day']} at {o['metadata']['upload_time']}")

print("\n" + "=" * 65)
print("\nSCRIPT OUTLINE -- Top Opportunity:")
print(opportunities[0]['outline'] if opportunities else "No topics found.")
print("""
===============================================================
NEXT STEPS:
  1. Pick the topic with the highest score above
  2. Use Claude to research the UNIQUE ANGLE
     (paste the CLAUDE RESEARCH PROMPT from the outline)
  3. Write your full Dark Files script (500+ words)
  4. Scroll down to Cell 8, paste your script
  5. Runtime -> Run all -> Download your finished video
===============================================================
""")


In [ ]:
# ── STEP 0: Verify GPU ──────────────────────────────────────────
import subprocess, os, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    print("❌  No GPU found!")
    print("👉  Go to Runtime → Change runtime type → GPU → T4 → Save")
    print("    Then restart and run again.")
    sys.exit(1)

# Print GPU name and memory
for line in result.stdout.split('\n'):
    if 'Tesla' in line or 'T4' in line or 'A100' in line or 'L4' in line or 'V100' in line:
        print(f"✅  GPU: {line.strip()}")
        break
else:
    print("✅  GPU detected")
    print(result.stdout[:300])

# Create working directories
for d in ['/content/dark_files/clips', '/content/dark_files/audio',
          '/content/dark_files/final', '/content/dark_files/cache']:
    os.makedirs(d, exist_ok=True)

print("\n📁  Working directories ready")
print("🚀  Ready to generate Dark Files videos!")


In [ ]:
# ── STEP 1: Install dependencies ────────────────────────────────
print("📦  Installing packages (3–5 min first time)...")

import subprocess
pkgs = [
    "edge-tts",
    "diffusers>=0.31.0",
    "transformers>=4.40.0",
    "accelerate",
    "imageio[ffmpeg]",
    "moviepy",
    "scipy",
    "sentencepiece",
    "protobuf",
]
subprocess.run(
    ["pip", "install", "-q", "--upgrade"] + pkgs,
    check=True
)

# Verify FFmpeg
r = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print(f"✅  {r.stdout.splitlines()[0]}" if r.returncode == 0 else "❌  FFmpeg missing")
print("✅  All packages installed!")


In [ ]:
# ── STEP 2: Imports ─────────────────────────────────────────────
import os, re, json, asyncio, subprocess, math, time, warnings
import numpy as np
from pathlib import Path

import torch
warnings.filterwarnings('ignore')

from IPython.display import Video, display, HTML
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16

if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"🖥️   GPU : {props.name}")
    print(f"💾  VRAM : {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠️   Running on CPU — video generation will be very slow")

print(f"\n✅  Ready  |  Device: {DEVICE.upper()}")


In [ ]:
# ── STEP 3: PASTE YOUR DARK FILES SCRIPT HERE ───────────────────
#
#  ✏️  Replace the example below with your full episode script.
#  Each paragraph becomes one scene.  Longer paragraph = longer clip.
# ─────────────────────────────────────────────────────────────────

EPISODE_TITLE = "The Classified Files"

YOUR_SCRIPT = """
In the summer of 1984, a small town in rural Ohio reported something that would be buried
for decades. Three witnesses saw lights hovering above the forest. By morning, six square
miles of trees were dead. The government arrived within hours.

The official explanation was a chemical spill from a nearby plant. But the plant had been
closed for two years. Not a single employee was interviewed. Not a single sample was tested.
The report was sealed before the week was out.

The local newspaper editor who first broke the story disappeared three weeks later. His files
were gone. His sources went silent. The case was never opened.

What really happened in Millbrook that night has never been explained. The documents we have
obtained tell a very different story — one the government has spent forty years trying to erase.
"""

# ── Voice options ────────────────────────────────────────────────
# Uncomment one voice (GuyNeural is the default Dark Files voice)
VOICE         = "en-US-GuyNeural"        # Deep authoritative male  ← DEFAULT
# VOICE       = "en-US-EricNeural"       # Serious male narrator
# VOICE       = "en-US-ChristopherNeural"# Calm, grave male voice
# VOICE       = "en-GB-RyanNeural"       # British male (BBC documentary feel)

SPEAKING_RATE  = "-10%"    # Slower = more dramatic
SPEAKING_PITCH = "-3Hz"    # Slightly deeper tone

# ─────────────────────────────────────────────────────────────────
words = len(YOUR_SCRIPT.split())
print(f"📺  Episode    : {EPISODE_TITLE}")
print(f"🎙️   Voice      : {VOICE}")
print(f"📝  Words      : {words}")
print(f"⏱️   Est. length: ~{words / (110/60):.0f} seconds  ({words / (110/60) / 60:.1f} min)")


In [ ]:
# ── STEP 4: Scene parser + Dark Files prompt engine ─────────────

# ── Dark Files visual prompt database ───────────────────────────
DARK_AESTHETIC = (
    "dark atmospheric cinematography, cold desaturated blue color palette, "
    "dramatic noir low-key lighting, deep crushed shadows, film grain texture, "
    "documentary thriller style, cinematic 16:9 widescreen, "
    "mysterious haunting atmosphere, ultra realistic, photorealistic, "
    "high contrast, professional cinematography"
)

DARK_NEGATIVE = (
    "bright sunny day, colorful cheerful, cartoon, animated, illustration, "
    "blurry, low quality, distorted, watermark, text overlay, logo, nsfw, "
    "happy atmosphere, over-exposed, white background, daytime"
)

VISUAL_MAP = [
    (r'forest|woods|trees|woodland',
     'dark misty forest at night, fog between ancient trees, faint moonlight'),
    (r'facility|complex|plant|factory|warehouse',
     'abandoned industrial facility at dusk, chain-link fence, security floodlights, concrete'),
    (r'laborator|lab|research|scientist',
     'dark sterile government laboratory corridor, sealed blast doors, cold fluorescent light'),
    (r'city|town|street|urban|neighborhood',
     'dark rain-slicked city street at night, distant neon reflections, empty pavement'),
    (r'ocean|sea|river|lake|water|coast',
     'dark turbulent water surface at night, storm clouds, cold dramatic moonlight'),
    (r'desert|nevada|arizona|wasteland|remote|plains',
     'remote desert landscape at dusk, dramatic storm clouds building, desolate empty road'),
    (r'prison|jail|detention|cell|incarcerat',
     'dark prison corridor at night, harsh single overhead light, iron bars casting long shadows'),
    (r'courtroom|trial|judge|lawyer|testimony|verdict',
     'dark courtroom interior, single overhead spotlight on witness stand, wooden benches, tension'),
    (r'church|chapel|cathedral|cemetery|grave|tombstone',
     'gothic fog-covered cemetery at midnight, crumbling headstones, cold moonlight through clouds'),
    (r'hospital|medical|morgue|autopsy|clinic',
     'abandoned hospital corridor, flickering fluorescent lights, peeling walls, cold sterile blue'),
    (r'police|detective|investig|crime scene|sheriff',
     'crime scene at night, yellow police tape in the wind, distant red-blue police lights in fog'),
    (r'document|file|report|classif|declassif|evidence|record',
     'close-up of classified government documents on a table, dramatic side-lighting, heavy black redactions'),
    (r'disappear|vanish|missing|abduct|gone',
     'empty dark room, single overturned chair, door ajar, single dim bulb swinging, eerie silence'),
    (r'government|military|pentagon|CIA|FBI|NSA|agency|federal|intelligence',
     'imposing government building at night, surveillance cameras, concrete facade, distant floodlights'),
    (r'witness|survivor|victim|family|mother|father|child|brother|sister',
     'shadowy silhouette of a person standing at a dark window at night, back-lit, motionless'),
    (r'secret|hidden|cover|buried|suppress|conceal',
     'heavy vault door in darkness, single flashlight beam, rusted locks, deep shadows'),
    (r'helicopter|aircraft|military aircraft|jet|plane',
     'military helicopters flying at night with search beams, dark overcast sky, rotor motion blur'),
    (r'phone|call|wiretap|surveillance|listen|intercept',
     'vintage rotary phone on a dark desk, reel-to-reel recording equipment, sinister single lamp'),
    (r'newspaper|media|press|headline|journalist|editor|broadcast',
     'dark newspaper archive room, stacked yellowed papers under a single lamp, vintage film aesthetic'),
    (r'night|midnight|3.?am|dark|after dark|evening|dusk|dawn',
     'deep night environment, minimal isolated light sources, heavy fog, mysterious still atmosphere'),
    (r'road|highway|bridge|tunnel|path|route',
     'empty dark highway stretching to infinity at night, faint headlights in the distance, fog'),
    (r'19[0-9]{2}|20[0-2][0-9]|decade|era|century|year',
     'cinematic historical recreation, period-appropriate environment, dramatic vintage film grain'),
    (r'body|remains|skeleton|bones|buried|grave',
     'dark woodland clearing at night, disturbed earth, police forensic flashlights in the dark'),
    (r'explosion|fire|burn|smoke|destroy|demolish',
     'smoldering ruins at night, distant flames reflecting on wet ground, emergency lights in smoke'),
]

def parse_scenes(script, max_words=55):
    paras = [p.strip() for p in script.strip().split('\n') if p.strip()]
    scenes = []
    for para in paras:
        if len(para.split()) <= max_words:
            scenes.append(para)
        else:
            sents = re.split(r'(?<=[.!?])\s+', para)
            bucket, bw = [], 0
            for s in sents:
                sw = len(s.split())
                if bw + sw > max_words and bucket:
                    scenes.append(' '.join(bucket))
                    bucket, bw = [s], sw
                else:
                    bucket.append(s); bw += sw
            if bucket:
                scenes.append(' '.join(bucket))
    return [s for s in scenes if s.strip()]

def make_prompt(scene_text):
    low = scene_text.lower()
    matched = []
    for pattern, visual in VISUAL_MAP:
        if re.search(pattern, low, re.IGNORECASE):
            matched.append(visual)
    if matched:
        base = ', '.join(matched[:2])
    else:
        base = 'dark mysterious empty environment, dramatic single light source, heavy shadows, fog'
    return f"{base}, {DARK_AESTHETIC}", DARK_NEGATIVE

def estimate_secs(text, wpm=110):
    return (len(text.split()) / wpm) * 60

# ── Preview ──────────────────────────────────────────────────────
scenes = parse_scenes(YOUR_SCRIPT)
print(f"📊  Parsed into {len(scenes)} scenes\n{'='*65}")
for i, s in enumerate(scenes):
    dur = estimate_secs(s)
    prompt, _ = make_prompt(s)
    print(f"\n🎬  Scene {i+1}  ({dur:.1f}s | {len(s.split())} words)")
    print(f"    Text   : {s[:75]}...")
    print(f"    Prompt : {prompt[:80]}...")
total = sum(estimate_secs(s) for s in scenes)
print(f"\n{'='*65}")
print(f"⏱️   Total estimated duration: {total:.0f}s  ({total/60:.1f} min)")
print(f"⏳  Estimated generation time on T4: {len(scenes)*3:.0f}–{len(scenes)*5:.0f} min")


In [ ]:
# ── STEP 5: Generate voiceover + caption timing ──────────────────
import nest_asyncio
nest_asyncio.apply()
import edge_tts, asyncio, json, subprocess

async def gen_voice_simple(text, voice, rate, pitch, audio_out):
    communicate = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
    await communicate.save(audio_out)

def audio_dur(path):
    r = subprocess.run(
        ['ffprobe','-v','quiet','-print_format','json','-show_format', path],
        capture_output=True, text=True
    )
    return float(json.loads(r.stdout)['format']['duration'])

print(f"🎙️   Generating voiceover  ({VOICE})...\n")
audio_paths, vtt_paths, durations = [], [], []

for i, scene in enumerate(scenes):
    ap = f'/content/dark_files/audio/scene_{i:03d}.mp3'
    vp = f'/content/dark_files/audio/scene_{i:03d}.vtt'
    asyncio.run(gen_voice_simple(scene, VOICE, SPEAKING_RATE, SPEAKING_PITCH, ap))
    d = audio_dur(ap)
    with open(vp, 'w') as f:
        f.write(f"WEBVTT\n\n00:00.000 --> {int(d//60):02d}:{d%60:06.3f}\n{scene}\n\n")
    audio_paths.append(ap); vtt_paths.append(vp); durations.append(d)
    print(f"  ✅  Scene {i+1}/{len(scenes)}: {d:.2f}s  |  {scene[:55]}...")

total_duration = sum(durations)
print(f"\n✅  Voiceover complete!")
print(f"⏱️   Total duration: {total_duration:.1f}s  ({total_duration/60:.1f} min)")


In [ ]:
# ── STEP 6: Load LTX-Video model ────────────────────────────────
#
#  First run downloads ~4 GB — this is a one-time download per session.
#  Subsequent runs in the same session reuse the cache.
# ─────────────────────────────────────────────────────────────────
import torch
from diffusers import LTXPipeline
from diffusers.utils import export_to_video

torch.cuda.empty_cache()
print("⬇️   Loading LTX-Video (Lightricks) ...\n"
      "    First run downloads ~4 GB — please wait.\n")

pipe = LTXPipeline.from_pretrained(
    "Lightricks/LTX-Video",
    torch_dtype=torch.bfloat16,
    cache_dir='/content/dark_files/cache',
)
pipe.enable_model_cpu_offload()   # keeps VRAM usage low on T4

print(f"\n✅  LTX-Video loaded!")
print(f"💾  VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [ ]:
# ── STEP 7: Generate video clips ────────────────────────────────
import time
from diffusers.utils import export_to_video

def valid_frames(secs, fps=25):
    """Return a valid LTX-Video frame count (8k+1) for the given duration."""
    n = max(25, int(secs * fps))
    n = ((n - 1) // 8) * 8 + 1
    return min(n, 257)

def gen_clip(prompt, neg, n_frames, out_path, seed=0):
    gen = torch.Generator(device='cuda').manual_seed(seed)
    out = pipe(
        prompt=prompt,
        negative_prompt=neg,
        width=704, height=480,
        num_frames=n_frames,
        num_inference_steps=30,
        guidance_scale=3.5,
        generator=gen,
    )
    frames = out.frames[0]
    export_to_video(frames, out_path, fps=25)
    return out_path

def trim_to_duration(src, dst, duration):
    """Trim/pad clip to match exact audio duration."""
    # Get actual clip duration
    r = subprocess.run(
        ['ffprobe','-v','quiet','-print_format','json','-show_format', src],
        capture_output=True, text=True
    )
    clip_dur = float(json.loads(r.stdout)['format']['duration'])

    if clip_dur >= duration:
        # Trim
        subprocess.run([
            'ffmpeg','-i', src,'-t', str(duration),
            '-c:v','libx264','-crf','18','-preset','fast',
            '-pix_fmt','yuv420p', dst, '-y'
        ], capture_output=True, check=True)
    else:
        # Loop to fill
        loops = math.ceil(duration / clip_dur)
        subprocess.run([
            'ffmpeg',
            '-stream_loop', str(loops),
            '-i', src,
            '-t', str(duration),
            '-c:v','libx264','-crf','18','-preset','fast',
            '-pix_fmt','yuv420p', dst, '-y'
        ], capture_output=True, check=True)

print(f"🎬  Generating {len(scenes)} video clips ...\n{'='*65}")
clip_paths = []
t0 = time.time()

for i, (scene, dur) in enumerate(zip(scenes, durations)):
    raw   = f'/content/dark_files/clips/clip_{i:03d}_raw.mp4'
    final = f'/content/dark_files/clips/clip_{i:03d}.mp4'
    prompt, neg = make_prompt(scene)
    n_frames    = valid_frames(dur)

    print(f"\n🎥  Clip {i+1}/{len(scenes)}  ({dur:.1f}s | {n_frames} frames)")
    print(f"    {prompt[:85]}...")

    t1 = time.time()
    gen_clip(prompt, neg, n_frames, raw, seed=i * 17 + 42)
    trim_to_duration(raw, final, dur)
    clip_paths.append(final)

    elapsed = time.time() - t1
    done    = i + 1
    eta     = (time.time() - t0) / done * (len(scenes) - done)
    print(f"    ✅  {elapsed:.0f}s  |  ETA: ~{eta/60:.0f} min remaining")

print(f"\n✅  All clips generated!  Total: {(time.time()-t0)/60:.1f} min")


In [ ]:
# ── STEP 8: Generate dark ambient background music ───────────────
from transformers import pipeline as hf_pipeline
import scipy.io.wavfile as wav

# Free GPU memory from video model first
del pipe
torch.cuda.empty_cache()

print("🎵  Loading MusicGen-small ...")
music_pipe = hf_pipeline(
    'text-to-audio',
    'facebook/musicgen-small',
    device=0 if DEVICE == 'cuda' else -1,
)

MUSIC_PROMPT = (
    "dark ambient music, mysterious thriller documentary, tense suspenseful atmosphere, "
    "slow deep bass drones, ominous low-frequency hum, haunting cinematic orchestral score, "
    "psychological thriller, minimal slow tempo, no vocals, instrumental only, eerie"
)

print("🎵  Generating 30-second dark ambient loop ...")
music = music_pipe(MUSIC_PROMPT, forward_params={"do_sample": True, "max_new_tokens": 1500})

raw_wav = '/content/dark_files/audio/music_raw.wav'
wav.write(raw_wav, music[0]['sampling_rate'], music[0]['audio'].T)

# Loop to match full video duration + fade out
music_wav = '/content/dark_files/audio/music_final.wav'
fade_start = max(0, total_duration - 4)
subprocess.run([
    'ffmpeg', '-stream_loop', '-1', '-i', raw_wav,
    '-t', str(total_duration + 2),
    '-af', f'afade=t=out:st={fade_start:.1f}:d=4',
    '-c:a', 'pcm_s16le', music_wav, '-y'
], capture_output=True, check=True)

del music_pipe
torch.cuda.empty_cache()
print(f"✅  Music ready  ({total_duration:.0f}s, fades out at end)")


In [ ]:
# ── STEP 9: Assemble clips with fade transitions ─────────────────

def add_fades(src, dst, dur, fade_dur=0.4):
    """Add fade-in + fade-out to a clip."""
    fade_out_start = max(0, dur - fade_dur)
    subprocess.run([
        'ffmpeg', '-i', src,
        '-vf', (f'fade=t=in:st=0:d={fade_dur},'
                f'fade=t=out:st={fade_out_start:.2f}:d={fade_dur}'),
        '-c:v', 'libx264', '-crf', '18', '-preset', 'fast',
        '-pix_fmt', 'yuv420p', dst, '-y'
    ], capture_output=True, check=True)

print("✂️   Adding fade transitions to each clip ...")
faded_paths = []
for i, (cp, dur) in enumerate(zip(clip_paths, durations)):
    fp = f'/content/dark_files/clips/clip_{i:03d}_faded.mp4'
    add_fades(cp, fp, dur)
    faded_paths.append(fp)
    print(f"  ✅  Clip {i+1}/{len(scenes)} — fade in/out applied")

# Concatenate all faded clips
concat_list = '/content/dark_files/concat.txt'
with open(concat_list, 'w') as f:
    for p in faded_paths:
        f.write(f"file '{p}'\n")

raw_video = '/content/dark_files/final/video_raw.mp4'
subprocess.run([
    'ffmpeg', '-f', 'concat', '-safe', '0', '-i', concat_list,
    '-c:v', 'libx264', '-crf', '17', '-preset', 'fast', '-pix_fmt', 'yuv420p',
    raw_video, '-y'
], capture_output=True, check=True)
print("\n✅  Clips concatenated")

# Concatenate all voiceover segments
audio_list = '/content/dark_files/audio_concat.txt'
with open(audio_list, 'w') as f:
    for p in audio_paths:
        f.write(f"file '{p}'\n")

full_voice = '/content/dark_files/audio/voice_full.mp3'
subprocess.run([
    'ffmpeg', '-f', 'concat', '-safe', '0', '-i', audio_list,
    '-c:a', 'copy', full_voice, '-y'
], capture_output=True, check=True)

# Merge video + voiceover
video_with_voice = '/content/dark_files/final/video_voiced.mp4'
subprocess.run([
    'ffmpeg', '-i', raw_video, '-i', full_voice,
    '-map', '0:v', '-map', '1:a',
    '-c:v', 'copy', '-c:a', 'aac', '-b:a', '192k',
    '-shortest', video_with_voice, '-y'
], capture_output=True, check=True)
print("✅  Voiceover merged")


In [ ]:
# ── STEP 10: Apply Dark Files cinematic color grade ──────────────

print("🎨  Applying Dark Files color grade ...")
print("    → Cold blue channel shift")
print("    → Crushed blacks + compressed highlights")
print("    → Desaturated tones (0.65 saturation)")
print("    → High contrast (1.3)")
print("    → Subtle film grain")

DARK_FILES_GRADE = ",".join([
    # Cold blue shift: reduce red slightly, boost blue
    "colorchannelmixer=rr=0.82:rg=0.01:rb=0.0:gr=0.0:gg=0.88:gb=0.04:br=0.0:bg=0.05:bb=1.1",
    # S-curve: crush blacks, compress highlights
    "curves=all='0/0 0.18/0.08 0.5/0.44 0.82/0.72 1/0.87'",
    # Desaturate + contrast boost + slight darken
    "eq=saturation=0.65:contrast=1.3:brightness=-0.04:gamma=1.05",
    # Subtle film grain
    "noise=alls=3:allf=t+u",
    # Gentle detail sharpening
    "unsharp=5:5:0.35:3:3:0.0"
])

video_graded = '/content/dark_files/final/video_graded.mp4'
subprocess.run([
    'ffmpeg', '-i', video_with_voice,
    '-vf', DARK_FILES_GRADE,
    '-c:v', 'libx264', '-crf', '17', '-preset', 'slow',
    '-c:a', 'copy',
    video_graded, '-y'
], check=True, capture_output=True)

print("\n✅  Dark Files color grade applied!")


In [ ]:
# ── STEP 11: Build + burn synced captions ───────────────────────

def vtt_time_to_ms(t):
    t = t.strip().replace(',','.')
    parts = t.split(':')
    if len(parts) == 3:
        h, m, s = parts
        return (int(h)*3600 + int(m)*60 + float(s)) * 1000
    m, s = parts
    return (int(m)*60 + float(s)) * 1000

def ms_to_srt(ms):
    ms = int(ms)
    h  = ms // 3_600_000; ms %= 3_600_000
    m  = ms // 60_000;    ms %= 60_000
    s  = ms // 1_000;     ms %= 1_000
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

print("📝  Building synced SRT caption file ...")
entries, idx, offset_ms = [], 1, 0

for vp, dur in zip(vtt_paths, durations):
    try:
        with open(vp, encoding='utf-8') as f:
            content = f.read()
        for block in re.split(r'\n{2,}', content.strip()):
            if '-->' not in block:
                continue
            lines = block.strip().splitlines()
            t_line = next((l for l in lines if '-->' in l), None)
            if not t_line:
                continue
            text = ' '.join(
                l for l in lines
                if '-->' not in l and not l.strip().isdigit() and l.strip()
            )
            if not text:
                continue
            s_str, e_str = t_line.split('-->')
            s_ms = vtt_time_to_ms(s_str) + offset_ms
            e_ms = vtt_time_to_ms(e_str) + offset_ms
            entries.append((idx, ms_to_srt(s_ms), ms_to_srt(e_ms), text))
            idx += 1
    except Exception as ex:
        print(f"  ⚠️   VTT parse warning for {vp}: {ex}")
    offset_ms += dur * 1000

srt_path = '/content/dark_files/audio/captions.srt'
with open(srt_path, 'w', encoding='utf-8') as f:
    for n, s, e, t in entries:
        f.write(f"{n}\n{s} --> {e}\n{t}\n\n")
print(f"  ✅  {len(entries)} caption entries generated")

# Burn captions with Dark Files styling
CAPTION_STYLE = (
    "FontName=Arial,Bold=1,FontSize=17,"
    "PrimaryColour=&H00FFFFFF,"     # white text
    "OutlineColour=&H00000000,"     # black outline
    "BackColour=&H55000000,"        # semi-transparent black box
    "BorderStyle=3,Outline=1.5,Shadow=0,"
    "Alignment=2,MarginV=28"        # bottom centre, 28px margin
)

video_captioned = '/content/dark_files/final/video_captioned.mp4'
subprocess.run([
    'ffmpeg', '-i', video_graded,
    '-vf', f"subtitles={srt_path}:force_style='{CAPTION_STYLE}'",
    '-c:v', 'libx264', '-crf', '17',
    '-c:a', 'copy',
    video_captioned, '-y'
], check=True, capture_output=True)

print("✅  Captions burned in!")


In [ ]:
# ── STEP 12: Final audio mix (voice 100% + music 15%) ───────────

print("🎵  Mixing voiceover + background music ...")

safe_title  = re.sub(r'[^\w\s-]', '', EPISODE_TITLE).strip().replace(' ', '_')
final_video = f'/content/dark_files/final/DARK_FILES_{safe_title}.mp4'

subprocess.run([
    'ffmpeg',
    '-i', video_captioned,
    '-i', music_wav,
    '-filter_complex',
    '[0:a]volume=1.0[v];[1:a]volume=0.14[m];[v][m]amix=inputs=2:duration=first:dropout_transition=2[mix]',
    '-map', '0:v', '-map', '[mix]',
    '-c:v', 'copy',
    '-c:a', 'aac', '-b:a', '192k',
    '-shortest',
    final_video, '-y'
], check=True, capture_output=True)

# ── Final stats ──────────────────────────────────────────────────
r = subprocess.run(
    ['ffprobe','-v','quiet','-print_format','json','-show_format','-show_streams', final_video],
    capture_output=True, text=True
)
info = json.loads(r.stdout)
size_mb  = int(info['format']['size']) / (1024*1024)
vid_dur  = float(info['format']['duration'])
v_stream = next((s for s in info['streams'] if s['codec_type']=='video'), {})
width    = v_stream.get('width', 704)
height   = v_stream.get('height', 480)

print(f"\n{'='*55}")
print(f"🎉  DARK FILES VIDEO COMPLETE!")
print(f"{'='*55}")
print(f"📁  File     : {os.path.basename(final_video)}")
print(f"⏱️   Duration : {vid_dur:.1f}s  ({vid_dur/60:.1f} min)")
print(f"📊  Size     : {size_mb:.1f} MB")
print(f"🎬  Res.     : {width}x{height} @ 25fps")
print(f"🎨  Grade    : Dark Files cinematic (cold blue, high contrast)")
print(f"🎙️   Voice    : {VOICE}")
print(f"📝  Captions : {len(entries)} synced entries")
print(f"🎵  Music    : Dark ambient (AI-generated, royalty-free)")
print(f"{'='*55}")


In [ ]:
# ── STEP 13: Preview + download ──────────────────────────────────

print("🎬  Loading preview ...\n")
display(Video(final_video, width=704, height=480, embed=True))

print("\n📥  Downloading your Dark Files video ...")
if IN_COLAB:
    colab_files.download(final_video)
else:
    print(f"   File saved at: {final_video}")

print("""
✅  YOUR VIDEO IS READY FOR YOUTUBE

YouTube Upload Checklist:
  ✅  MP4 container  (H.264 + AAC)
  ✅  16:9 aspect ratio
  ✅  25 fps
  ✅  192 kbps stereo audio
  ✅  Synced captions burned in
  ✅  Dark Files cinematic color grade
  ✅  Professional neural voiceover
  ✅  Dark ambient background music (royalty-free)

💡  Pro tips after upload:
  • Add a custom thumbnail in YouTube Studio
  • Paste your script as the video description for SEO
  • Add chapter markers manually for longer episodes
""")
